### Spin Test 
This script projects data to surface space and performs a spin test

#### Import Packages

In [54]:
import numpy as np
import pandas as pd
import netneurotools.stats as stats
from neuromaps.datasets import fetch_atlas
from nilearn.datasets import fetch_atlas_schaefer_2018
from netneurotools.datasets import fetch_schaefer2018
import neuromaps
import nibabel as nib
from neuromaps.parcellate import Parcellater
from neuromaps import transforms
from neuromaps import nulls
from scipy.stats import pearsonr, spearmanr
from nilearn.plotting import plot_surf
from nilearn.surface import load_surf_data
from nilearn import surface, plotting, datasets
from neuromaps import datasets, images, nulls, resampling, stats
import netneurotools.datasets
from netneurotools.datasets import fetch_cammoun2012

#### Set Paths and Variables

In [52]:
%reset

Once deleted, variables cannot be recovered. Proceed (y/[n])?  y


In [56]:
dataset = 'longglucest_outputmeasures2'
atlas = 'atl-Cammoun2012_res-500' # 'Schaefer2018_1000Parcels_17Networks' #  #   # # 
nmaps = ["NMDA", "mGluR5", "GABA"]
maps = ["cest", "NMDA", "mGluR5", "GABA"]
#group = "PSY" # Only analyze this hstatus group
#group = ["PSY","HC"]
n_permute = 10000

# Set paths
inpath = "/Users/pecsok/Desktop/ImageData/PMACS_remote/data/nmaps/analyses/" + atlas
outpath = "/Users/pecsok/Desktop/ImageData/PMACS_remote/data/nmaps/analyses/" + atlas

# Read in data
#receptor_df = pd.read_csv("/Users/pecsok/projects/Neuromaps/pecsok_pfns/neuromaps/results/receptor_data_scale1000_17.csv", sep=',')
means= pd.read_csv(inpath + '/means_std_' + dataset + '_' + atlas + '.csv', sep=',') #subjectnmaps

# Reformat data 
means["Parcel"] = means["Parcel"].str.replace(' NZMean', '', regex=False)
if 'group' in locals(): # Filter by hstatus group of interest if desired
    means = means[means['hstatus']== group] 
means.set_index('Parcel', inplace = True)
#print(means)
"""
receptor_df = receptor_df.reset_index(drop=True)
receptor_df.rename(columns={'GABAa': 'GABA'}, inplace=True)
receptor_df.index = pd.RangeIndex(start=1, stop=len(receptor_df) + 1, step=1)
print(means)
print(receptor_df)
"""
if atlas == 'Schaefer2018_1000Parcels_17Networks':
    receptor_df = pd.read_csv("/Users/pecsok/projects/Neuromaps/pecsok_pfns/neuromaps/results/receptor_data_scale1000_17.csv", sep=',')
    receptor_df = receptor_df.reset_index(drop=True)
    receptor_df.rename(columns={'GABAa': 'GABA'}, inplace=True)
    receptor_df.index = pd.RangeIndex(start=1, stop=len(receptor_df) + 1, step=1)
    # Read in surface atlases
    schaefer_cifti_path = '/Users/pecsok/neuromaps-data/atlases/fsLR/Schaefer2018_900Parcels_17Networks_order.dlabel.nii'
    annot_rh_schaefer = '/Users/pecsok/neuromaps-data/atlases/fsaverage/lh.Schaefer2018_1000Parcels_17Networks_order.annot'
    annot_lh_schaefer = '/Users/pecsok/neuromaps-data/atlases/fsaverage/rh.Schaefer2018_1000Parcels_17Networks_order.annot'
    # Fetch atlas labels
    schaefer = fetch_atlas_schaefer_2018(n_rois=1000, yeo_networks=17)
    labels = schaefer['labels']  # These are the MNI coordinates of the parcels
    labels = [label.decode('utf-8') for label in labels]

elif atlas == 'atl-Cammoun2012_res-500':
    scale = 'cammoun2012_scale500' #'scale1000_17' #'Harvard_Oxford_sub' 
    # Import surface data
    cammoun = netneurotools.datasets.fetch_cammoun2012(version='fsaverage')
    cammoun = cammoun['scale500']
    annot_lh = cammoun[0]
    annot_rh = cammoun[1]
    # Import receptor_df
    receptor_df = pd.read_csv("/Users/pecsok/projects/Neuromaps/pecsok_pfns/neuromaps/results/receptor_data_cammoun2012_scale500.csv", sep=',')
    receptor_df.rename(columns={'GABAa': 'GABA'}, inplace=True)   
    receptor_df.index = pd.RangeIndex(start=1, stop=len(receptor_df) + 1, step=1)
    # Import labels and remove subcortex
    cam = fetch_cammoun2012()
    info = pd.read_csv(cam['info'], sep=',')
    info = info[info['scale']== 'scale500']
    #labels = info['label'][info['structure']=='cortex'].values
    hemisphere = info['hemisphere'][info['structure']=='cortex'].values
    labels = info['label'][info['structure']=='cortex'].values + hemisphere
    print(labels.shape)
    # Remove subcortex from other dfs
    receptor_df = receptor_df[receptor_df['Parcel'].str.contains('|'.join(labels))]
    print(receptor_df.shape)
    means = means[means.index.str.contains('|'.join(labels))]
    print(means.shape)

(1000,)
(1000, 4)
(104, 6)


### Functions

In [58]:
def calculate_null_corrs(null_dfs, nmaps):
    """
    Calculates correlation values for each null df and stores in output df
    Args:
    null_dfs (list): permutation matrices with new indices
    spin_df (pd.DataFrame): DataFrame where each column contains scrambled row indices (1-based).
    Returns:
    null_dfs (list): List of DataFrames with rows reordered according to scrambled indices.
    """
    # Initialize an empty dataframe to store the results
    columns = ['shift_iteration']
    for col in nmaps:
        columns.append(f'{col}_r')
        columns.append(f'{col}_p')
    
    null_stats_df = pd.DataFrame(columns=columns)
    
    # Loop through each null dataframe (each shift iteration)
    for i, null_df in enumerate(null_dfs):
        result_row = {'permutation': i + 1}
        
        # For each nmaps column, calculate r and p values with respect to CESTavg
        for col in nmaps:
            r_value, p_value = pearsonr(null_df['CEST_avg'], null_df[col])
            result_row[f'{col}_r'] = r_value
            result_row[f'{col}_p'] = p_value
        
        # Convert result_row to a DataFrame and concatenate it
        result_row_df = pd.DataFrame([result_row])
        null_stats_df = pd.concat([null_stats_df, result_row_df], ignore_index=True)
    
    return null_stats_df
########################

def calculate_real_correlations(df, nmaps):
    """
    This function calculates pearson correlations between CEST_avg and nmap columns.
    Args:
    df (pd.DataFrame): df with cest and nmap parcelwise data
    nmaps (list): names of nmaps
    Returns: 
    Correlation values and p values for parametric associations.
    """
    real_corrs = {}
    for col in nmaps:
        r_value, _ = pearsonr(df['CEST_avg'], df[col])
        real_corrs[col] = r_value
    return real_corrs
#####################

def calculate_empirical_pvalues(real_corrs, null_stats_df, nmaps):
    """
    This function compares real correlations to the null distribution to calculate empirical p-values
    Args:
    real_corrs (pd.DataFrame): df with r and p values for cest-nmap associations
    null_stats_df (pd.DataFrame): df with r and p value for cest-nmap associations per iteration
    nmaps (list): names of nmaps
    Returns: 
    Empirical p values based on null model
    """    
    p_values = {}
    
    # Loop through each column in nmaps
    for col in nmaps:
        # Extract the null r-values for this nmaps column
        null_r_values = null_stats_df[f'{col}_r']
        
        # Calculate the empirical p-value: proportion of null r-values more extreme than the real r-value
        real_r = real_corrs[col]
        
        # Golia's addition - correct for if r values aren't centered on zero:
        null_mean = null_r_values.mean() # Find mean of r nulls
        # Subtract mean from null and real r values.
        null_r_values = null_r_values - null_mean 
        real_r = real_r - null_mean
        
        # Two-tailed test: count both large positive and negative extremes
        extreme_count = np.sum(np.abs(null_r_values) >= np.abs(real_r))
        
        # Compute the empirical p-value
        p_empirical = extreme_count / len(null_r_values)
        p_values[col] = round(p_empirical, 15)
    
    return p_values

### Generate permutations

In [59]:
# Convert fsavg annotations to gifti
fsgifti = images.annot_to_gifti((annot_lh, annot_rh))
# Spin indices
spin_index = nulls.vazquez_rodriguez(data=None, atlas='fsaverage', density='164k', # Fix double check density '164k'
                                     n_perm=n_permute, seed=1234, parcellation=fsgifti)


In [60]:
# Compile null dfs
test = np.array(receptor_df.iloc[:, 1:])
null = test[spin_index]
null_dfs = list()
for row_index in range(null.shape[1]):  
    spindf = pd.DataFrame(null[:, row_index, :], columns=['NMDA', 'mGluR5', 'GABA'])
    spindf['Parcel'] = labels
    null_dfs.append(spindf)
print(len(null_dfs))

10000


In [61]:
# Keep only parcels in slice
means_cest = means[['hstatus', 'CEST_avg']]
print(means_cest)
merged_dfs = []
#print(means_cest)
for null_df in null_dfs:
    merged_df = means_cest.merge(null_df, on='Parcel', how='inner')
    merged_dfs.append(merged_df)
print(merged_dfs[0])


                           hstatus  CEST_avg
Parcel                                      
caudalanteriorcingulate_5R      HC  0.140048
caudalanteriorcingulate_5R     PSY -0.011085
cuneus_3R                       HC  0.884299
cuneus_3R                      PSY  0.305783
cuneus_5R                       HC  1.411665
...                            ...       ...
superiorfrontal_7R             PSY -0.455633
superiorfrontal_8R              HC -0.857712
superiorfrontal_8R             PSY -0.800876
superiorfrontal_9R              HC -0.154002
superiorfrontal_9R             PSY -0.372532

[104 rows x 2 columns]
                         Parcel hstatus  CEST_avg      NMDA    mGluR5  \
0    caudalanteriorcingulate_5R      HC  0.140048  1.024301  0.830245   
1    caudalanteriorcingulate_5R     PSY -0.011085  1.024301  0.830245   
2                     cuneus_3R      HC  0.884299 -0.539328  0.194108   
3                     cuneus_3R     PSY  0.305783 -0.539328  0.194108   
4                     cune

In [62]:
# Calculate stats of null distributions
null_stats_df = calculate_null_corrs(merged_dfs, nmaps)

real_corrs = calculate_real_correlations(means, nmaps)
empirical_p_values = calculate_empirical_pvalues(real_corrs, null_stats_df, nmaps)

print(empirical_p_values)

/var/folders/kk/w6xmtt2d55xfbxvb6wcqhq580000gp/T/ipykernel_89262/363167931.py:30: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  null_stats_df = pd.concat([null_stats_df, result_row_df], ignore_index=True)


{'NMDA': 0.0116, 'mGluR5': 0.3133, 'GABA': 0.0001}


In [64]:
print("Pearson R values with real data:\n",real_corrs,"\n\nEmpirical p values based on ",n_permute, " permutations:")
for col, p_val in empirical_p_values.items():
    print(f'{col}: {p_val:.25f}') 

Pearson R values with real data:
 {'NMDA': 0.4794093831311077, 'mGluR5': 0.24549237724901335, 'GABA': 0.6757882541171423} 

Empirical p values based on  10000  permutations:
NMDA: 0.0115999999999999992006394
mGluR5: 0.3133000000000000229150032
GABA: 0.0001000000000000000047922


In [50]:
import numpy as np
from scipy.stats import norm

def fisher_z_test(r1, r2, n1, n2):
    # Fisher Z transformation
    z1 = 0.5 * np.log((1 + r1) / (1 - r1))
    z2 = 0.5 * np.log((1 + r2) / (1 - r2))
    
    # Standard error
    se = np.sqrt(1 / (n1 - 3) + 1 / (n2 - 3))
    
    # Z statistic
    z_stat = (z1 - z2) / se
    
    # Two-tailed p-value
    p_value = 2 * (1 - norm.cdf(abs(z_stat)))
    
    return z_stat, p_value

# Example usage
r_hc_nmda = 0.47552583 # Correlation coefficient in HC
r_psy_nmda = 0.4861826   # Correlation coefficient in PSY
r_hc_gaba = 0.6970255086 # Correlation coefficient in HC
r_psy_gaba = 0.65476845  # Correlation coefficient in PSY
n_hc = 36    # Sample size in HC
n_psy = 55   # Sample size in PSY

z_stat, p_value = fisher_z_test(r_hc_nmda, r_psy_nmda, n_hc, n_psy)
print(f"Z-statistic for NMDA: {z_stat}, p-value: {p_value}")
z_stat, p_value = fisher_z_test(r_hc_gaba, r_psy_gaba, n_hc, n_psy)
print(f"Z-statistic for NMDA: {z_stat}, p-value: {p_value}")


Z-statistic for NMDA: -0.062285207013442416, p-value: 0.9503357086961357
Z-statistic for NMDA: 0.3499760378779926, p-value: 0.72635668083412


In [66]:
# Fishers test for lobar comparisons

import numpy as np
from scipy.stats import norm

def fisher_z_test(r1, r2, n1, n2):
    # Fisher Z transformation
    z1 = 0.5 * np.log((1 + r1) / (1 - r1))
    z2 = 0.5 * np.log((1 + r2) / (1 - r2))
    
    # Standard error
    se = np.sqrt(1 / (n1 - 3) + 1 / (n2 - 3))
    
    # Z statistic
    z_stat = (z1 - z2) / se
    
    # Two-tailed p-value
    p_value = 2 * (1 - norm.cdf(abs(z_stat)))
    
    return z_stat, p_value

# Example usage
r_par_nmda = 0.6482948 # Correlation coefficient in HC
r_fro_nmda = 0.4300273  # Correlation coefficient in PSY
r_par_gaba = 0.771351 # Correlation coefficient in HC
r_fro_gaba = 0.08859911  # Correlation coefficient in PSY
n_par = 19   # Sample size in HC
n_fro = 29   # Sample size in PSY

z_stat, p_value = fisher_z_test(r_par_nmda, r_fro_nmda, n_par, n_fro)
print(f"Z-statistic for NMDA: {z_stat}, p-value: {p_value}")
z_stat, p_value = fisher_z_test(r_par_gaba, r_fro_gaba, n_par, n_fro)
print(f"Z-statistic for NMDA: {z_stat}, p-value: {p_value}")


Z-statistic for NMDA: 0.9832475360078208, p-value: 0.32548562426139815
Z-statistic for NMDA: 2.9420586798664785, p-value: 0.0032603813912401858
